# PySpark left join

## Source data

The source comes from jupyter-pyspark/f1-sourcefiles

# Inner the data from the races.csv and circuits.csv


# Initalise a spark session

In [12]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import col

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


# Load races.csv and circuits.csv into dataframes

I filtered out circuits with circuitId between 70 and 79

In [13]:
# Load csv files into dataframes
#  Contains headers
# Infers schema



races = spark.read.csv("f1-sourcefiles/races.csv", header=True, inferSchema=True)


circuits = spark.read.csv("f1-sourcefiles/circuits.csv", header=True, inferSchema=True).filter("circuitId NOT BETWEEN 70 AND 79")

# Show the data type of the dataframes


In [14]:
# results 

races.printSchema()

# races

circuits.printSchema()

root
 |-- raceId: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- round: integer (nullable = true)
 |-- circuitId: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- date: date (nullable = true)
 |-- time: string (nullable = true)
 |-- url: string (nullable = true)
 |-- fp1_date: string (nullable = true)
 |-- fp1_time: string (nullable = true)
 |-- fp2_date: string (nullable = true)
 |-- fp2_time: string (nullable = true)
 |-- fp3_date: string (nullable = true)
 |-- fp3_time: string (nullable = true)
 |-- quali_date: string (nullable = true)
 |-- quali_time: string (nullable = true)
 |-- sprint_date: string (nullable = true)
 |-- sprint_time: string (nullable = true)

root
 |-- circuitId: integer (nullable = true)
 |-- circuitRef: string (nullable = true)
 |-- name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- country: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lng: double (nullable = true)
 |-- alt: int

# Use Alias in a LEFT JOIN

In [16]:
# left join on same column name i.e. circuitId


# Create and alias for a Dataframe just like table alias in SQL


races = races.alias("rcs")

circuits = circuits.alias("crts")


# LEFT join on circuitId 


joined_df = races.join(circuits, col("rcs.circuitId") == col("crts.circuitId"), how="left") \
    .filter("rcs.year BETWEEN 2023 AND 2025") \
    .select(col("rcs.raceId"), col("rcs.year").alias("race_year"), col("rcs.circuitId"), col("rcs.name").alias("race_name"), \
            col("crts.circuitId"), col("crts.name").alias("circuit_name"), col("crts.location"), col("crts.country") )

joined_df.explain(mode="formatted")

joined_df.show()

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- BroadcastHashJoin LeftOuter BuildRight (6)
      :- Filter (2)
      :  +- Scan csv  (1)
      +- BroadcastExchange (5)
         +- Filter (4)
            +- Scan csv  (3)


(1) Scan csv 
Output [4]: [raceId#632, year#633, circuitId#635, name#636]
Batched: false
Location: InMemoryFileIndex [file:/home/robyip/projects/pyspark-deltalake/jupyter-pyspark/f1-sourcefiles/races.csv]
PushedFilters: [IsNotNull(year), GreaterThanOrEqual(year,2023), LessThanOrEqual(year,2025)]
ReadSchema: struct<raceId:int,year:int,circuitId:int,name:string>

(2) Filter
Input [4]: [raceId#632, year#633, circuitId#635, name#636]
Condition : ((isnotnull(year#633) AND (year#633 >= 2023)) AND (year#633 <= 2025))

(3) Scan csv 
Output [4]: [circuitId#685, name#687, location#688, country#689]
Batched: false
Location: InMemoryFileIndex [file:/home/robyip/projects/pyspark-deltalake/jupyter-pyspark/f1-sourcefiles/circuits.csv]
PushedFilters: [Or(LessThan(circu

# Filter after the left join where circuits dataframe is NULL and pre-filter the races Dataframe

In [22]:
# left join on same column name i.e. circuitId


# Create and alias for a Dataframe just like table alias in SQL


races = races.alias("rcs")

circuits = circuits.alias("crts")


# LEFT join on circuitId 

# pre-filter races dataframe
# filter for only null after the LEFT JOIN
joined_df = races.filter("rcs.year BETWEEN 2023 AND 2025") \
    .join(circuits, col("rcs.circuitId") == col("crts.circuitId"), how="left") \
    .filter("crts.circuitId IS NULL") \
    .select(col("rcs.raceId"), col("rcs.year").alias("race_year"), col("rcs.circuitId"), col("rcs.name").alias("race_name"), \
            col("crts.circuitId"), col("crts.name").alias("circuit_name"), col("crts.location"), col("crts.country") )

joined_df.explain(mode="formatted")

joined_df.show()

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- Filter (7)
      +- BroadcastHashJoin LeftOuter BuildRight (6)
         :- Filter (2)
         :  +- Scan csv  (1)
         +- BroadcastExchange (5)
            +- Filter (4)
               +- Scan csv  (3)


(1) Scan csv 
Output [4]: [raceId#632, year#633, circuitId#635, name#636]
Batched: false
Location: InMemoryFileIndex [file:/home/robyip/projects/pyspark-deltalake/jupyter-pyspark/f1-sourcefiles/races.csv]
PushedFilters: [IsNotNull(year), GreaterThanOrEqual(year,2023), LessThanOrEqual(year,2025)]
ReadSchema: struct<raceId:int,year:int,circuitId:int,name:string>

(2) Filter
Input [4]: [raceId#632, year#633, circuitId#635, name#636]
Condition : ((isnotnull(year#633) AND (year#633 >= 2023)) AND (year#633 <= 2025))

(3) Scan csv 
Output [4]: [circuitId#685, name#687, location#688, country#689]
Batched: false
Location: InMemoryFileIndex [file:/home/robyip/projects/pyspark-deltalake/jupyter-pyspark/f1-sourcefiles/circuits.csv